# ENTERPRISE KNOWLEDGE MINING SYSTEM

#### DOCUMENT INGESTION

In [1]:
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import time

BASE_URL = "https://export.arxiv.org/api/query"

params = {
    "search_query": "all:computer",
    "start": 0,
    "max_results": 5
}

# Build query URL safely
query_url = BASE_URL + "?" + urllib.parse.urlencode(params)

request = urllib.request.Request(
    query_url,
    headers={
        "User-Agent": "arxiv-data-project/1.0 (research script)"
    }
)

time.sleep(3)

# Make request
with urllib.request.urlopen(request) as response:
    xml_data = response.read()

# Parse XML
root = ET.fromstring(xml_data)

# arXiv uses Atom namespace
ns = {
    "atom": "http://www.w3.org/2005/Atom"
}

papers = []

for entry in root.findall("atom:entry", ns):
    title = entry.find("atom:title", ns).text.strip()
    summary = entry.find("atom:summary", ns).text.strip()
    published = entry.find("atom:published", ns).text
    
    authors = [
        author.find("atom:name", ns).text
        for author in entry.findall("atom:author", ns)
    ]

    pdf_link = next((link.attrib.get("href") for link in entry.findall("atom:link", ns) 
                if link.attrib.get("title") == "pdf" and link.attrib.get("type") == "application/pdf" and link.attrib.get("rel") == "related"), 
                None
            )

    papers.append({
        "title": title,
        "authors": authors,
        "published": published,
        "summary": summary,
        "pdf_link": pdf_link
    })

# Display parsed results
for paper in papers:
    print("\nTITLE:", paper["title"])
    print("AUTHORS:", ", ".join(paper["authors"]))
    print("PUBLISHED:", paper["published"])
    print("SUMMARY:", paper["summary"][:200], "...")
    print("PDF LINK:", paper["pdf_link"])


TITLE: Vision Based Game Development Using Human Computer Interaction
AUTHORS: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
PUBLISHED: 2010-02-10T19:46:07Z
SUMMARY: A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long vo ...
PDF LINK: https://arxiv.org/pdf/1002.2191v1

TITLE: From truth to computability I
AUTHORS: Giorgi Japaridze
PUBLISHED: 2004-07-21T03:58:22Z
SUMMARY: The recently initiated approach called computability logic is a formal theory of interactive computation. See a comprehensive online source on the subject at http://www.cis.upenn.edu/~giorgi/cl.html . ...
PDF LINK: https://arxiv.org/pdf/cs/0407054v2

TITLE: Changing Neighbors k Secure Sum Protocol for Secure Multi Party Computation
AUTHORS: Rashid Sheikh, Beerendra Kumar, Durgesh Kumar Mishra
PUBLISHED: 2010-02-11T19:58:10Z
SUMMARY: Secure sum computation of private data inpu

#### EXTRACTING PDF AND CONVERTING TO MARKDOWN

In [2]:
# trying pymupdf4llm
import pymupdf4llm
import urllib.request
import os

pdf_dir = "pdfs"
os.makedirs(pdf_dir, exist_ok = True)

pdf_path = "https://arxiv.org/pdf/1002.2191v1"
local_pdf = os.path.join(pdf_dir, "Vision Based Game Development Using HCI.pdf")
urllib.request.urlretrieve(pdf_path, local_pdf)

pages = pymupdf4llm.to_markdown(local_pdf, page_chunks=True)

print(f"Pages extracted: {len(pages)}")
print(pages[0]["metadata"])
print(pages[0]["text"][:1000])


OCR disabled because Tesseract language data not found.
Pages extracted: 7
{'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}
_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ 

## Vision Based Game Development Using Human Computer Interaction 

Bharath  University                   St.Joseph College of Engineering                   Bharath University Chennai,India                                       Chennai,India                                     Chennai,India 

_**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is des

In [24]:
#checking for short/empty pages that might cause issues with NER
for page in pages:
    if len(page["text"].strip()) <50:
         print(f"Short/empty page: {page['metadata']}")

#checking the character count distribution
lengths = [len(p["text"]) for p in pages]
print(f'Average chars per page: {sum(lengths)/len(lengths):.0f}')
print(f"Min: {min(lengths)}, Max: {max(lengths)}")

Average chars per page: 3487
Min: 2493, Max: 4217


#### DATA CLEANING FOR CHUNKING

In [25]:
import re

def clean_text(text: str) -> str:
    if not text:
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove markdown heading markers and separator-only lines
    text = re.sub(r"(?m)^\s*#{1,6}\s*", " ", text)
    text = re.sub(r"(?m)^\s*[-*_~=`]{2,}\s*$", " ", text)

    # Remove hash symbols that can confuse NER
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)
    text = text.replace("#", " ")

    # Fix hyphenated line breaks: learn-\ning -> learning
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)

    # Remove obvious page-only lines
    text = re.sub(r"(?im)^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)

    # Keep lines as text for now, then flatten
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\n", " ")

    # Final whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [81]:
# Apply cleaning to all pages in pdf
clean_pages = []
for page in pages:
    cleaned = clean_text(page["text"])
    clean_pages.append({
        "text": cleaned,
        "metadata": page["metadata"]
    })

print("Pages before:", len(pages))
print("Pages after cleaning:", len(clean_pages))
print("Sample cleaned page text:", clean_pages[0]["text"][:500] if clean_pages else "N/A")

Pages before: 7
Pages after cleaning: 7
Sample cleaned page text: _(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ Vision Based Game Development Using Human Computer Interaction Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India _**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntar


In [85]:
#chunking the cleaned text for spacy using langchain text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for page in clean_pages:
    splits = text_splitter.split_text(page["text"])

    for s in splits:
        chunks.append({
            "text": s,
            "metadata": {
                "title": papers[0]["title"] if papers else "",
                "authors": papers[0]["authors"] if papers else [],
                "abstract": papers[0]["summary"] if papers else "",
                "creation_date": page["metadata"].get("creationDate", ""),
                "published": papers[0]["published"] if papers else "",
                "page_number": page["metadata"].get("page_number", ""),
                "page_count": page["metadata"].get("page_count", ""),
                "format": page["metadata"].get("format", "")
            }
        })

print(len(chunks))
print(chunks[0])

79
{'text': '_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ Vision Based Game Development Using Human Computer Interaction Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India _**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines', 'metadata': {'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': ['S. Sumathi', 'S. K. Srivatsa', 'M. Uma Maheswari'], 'abstract': 'A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer. Facial 

In [86]:
# separating metadata from text for NER to avoid noise
def text_for_ner(chunk):
    text = (chunk.get("text") or "")
    md = chunk.get("metadata", {})

    # Clean markdown/special chars that confuse NER
    text = re.sub(r"[*_`~^—–-]+", " ", text)

    if md.get("page_number") == 1:
        title = (md.get("title") or "")
        if title:
            text = re.sub(re.escape(title), " ", text, flags=re.IGNORECASE)
        
        # Remove author names from text
        for author in md.get("authors", []):
            if author:
                text = re.sub(rf"\b{re.escape(author)}\b", " ", text, flags=re.IGNORECASE)
        
        # Keep only text after Abstract
        parts = re.split(r"\babstract\b[:\-\s]*", text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            text = parts[1]

    return re.sub(r"\s+", " ", text).strip()

#### Entity Extraction

In [87]:
#using spacy model for NER

import spacy

nlp = spacy.load("en_core_web_md") # using at the moment since it has better NER performance than the small mode
# nlp = spacy.load("en_core_web_sm") # would eventually switch when increase research papers for faster performance

In [88]:
# updating metadata with NER results 
ner_texts = [text_for_ner(c) for c in chunks]
docs = nlp.pipe(ner_texts, batch_size=32)

for i, doc in enumerate(docs):
    chunks[i]["metadata"].update({
        "entities": [(ent.text, ent.label_) for ent in doc.ents],
        "ner_text": ner_texts[i]
    })

    if i < 5:
        print("\nChunk", i, "entities:", [("text:", ent.text, "label:", ent.label_) for ent in doc.ents][:100])
        print("Chunk", i, "NER text preview:", ner_texts[i][:180])


Chunk 0 entities: [('text:', 'Human Computer Interface', 'label:', 'ORG')]
Chunk 0 NER text preview: A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines

Chunk 1 entities: []
Chunk 1 NER text preview: . The system presented here is a vision based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This

Chunk 2 entities: []
Chunk 2 NER text preview: . Facial features (nose tip and eyes) are detected and tracked in real time to use their actions as mouse events. The coordinates and movement of the nose tip in the live video fee

Chunk 3 entities: [('text:', 'USB', 'label:', 'ORG'), ('text:', '30', 'label:', 'CARDINAL'), ('text:', 'second', 'label:', 'ORDINAL'), ('text:', 'Human Computer Interface', 'label:', 'ORG'), ('text:', 'HCI', 'label:', 'ORG'), ('text:', 'SSR Filter', 'label:', 'ORG'), ('text:', 'Hough', 'label:', 'PERSON'), ('text:

In [89]:
# printing the metadata of the first chunk to verify NER results are stored
print(chunks[0]["metadata"])

{'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': ['S. Sumathi', 'S. K. Srivatsa', 'M. Uma Maheswari'], 'abstract': 'A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer. Facial features (nose tip and eyes) are detected and tracked in realtime to use their actions as mouse events. The coordinates and movement of the nose tip in the live video feed are translated to become the coordinates and movement of the mouse pointer on the application. The left or right eye blinks fire left or right mouse click events. The system works with inexpensive USB cameras and runs at a frame rate of 30 frames per second.', 'creation_da

In [90]:
# final chunks for embedding
final_chunks = []

for i, chunk in enumerate(chunks):
    final_chunks.append({
        "text": chunk["text"],
        "metadata": {
            "title": papers[0]["title"] if papers else "",
            "authors": papers[0]["authors"] if papers else [],
            "abstract": papers[0]["summary"] if papers else "",
            "creation_date": chunk["metadata"].get("creationDate", ""),
            "published": papers[0]["published"] if papers else "",
            "page_number": chunk["metadata"].get("page_number", ""),
            "page_count": chunk["metadata"].get("page_count", ""),
            "format": chunk["metadata"].get("format", ""),
            "entities": chunk["metadata"].get("entities", [])
        }
    })

    if i < 5:
        print("\nFinal Chunk", i, "metadata:", final_chunks[-1]["metadata"])
        print("Final Chunk", i, "text preview:", final_chunks[-1]["text"][:180])


Final Chunk 0 metadata: {'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': ['S. Sumathi', 'S. K. Srivatsa', 'M. Uma Maheswari'], 'abstract': 'A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer. Facial features (nose tip and eyes) are detected and tracked in realtime to use their actions as mouse events. The coordinates and movement of the nose tip in the live video feed are translated to become the coordinates and movement of the mouse pointer on the application. The left or right eye blinks fire left or right mouse click events. The system works with inexpensive USB cameras and runs at a frame rate of 30 frames p

#### EMBEDDING WITH CHROMADB